In [1]:
using CSV
using DataFrames
using Dates
using Serialization

In [2]:
ENV["COLUMNS"] = 1000;

In [3]:
realdata = DataFrame(CSV.File("../data/jhhs_realdata_2021_02_18.csv"));

In [4]:
shortterm = DataFrame(CSV.File("../data/shortterm-2021-02-19.csv"));

In [5]:
longterm = DataFrame(CSV.File("../data/longterm-2021-01-24.csv"));

In [6]:
capacity = DataFrame(CSV.File("../data/jhhs_beds.csv"));

In [7]:
hospitals = sort(unique(realdata.hospital))
@show hospitals;

hospitals = ["BMC", "HCGH", "JHH", "SH", "SMH"]


In [8]:
function package_realdata()
    date_range = sort(unique(realdata.date))
    
    d_dict = Dict((row.hospital, row.date) => row for row in eachrow(realdata))
    
    active_total         = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].active_total         : 0 for h in hospitals, t in date_range]
    active_total_flagged = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].active_total_flagged : 0 for h in hospitals, t in date_range]
    active_icu           = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].active_icu           : 0 for h in hospitals, t in date_range]
    active_acute         = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].active_acute         : 0 for h in hospitals, t in date_range]
    admitted_total       = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].admissions_total     : 0 for h in hospitals, t in date_range]
    
    pkg = Dict()
    pkg[:total] = (active=active_total, admitted=admitted_total)
    pkg[:icu] = (active=active_icu,)
    pkg[:acute] = (active=active_acute,)
    pkg[:total_flagged] = (active=active_total_flagged,)
    pkg[:meta] = (;hospitals, date_range)
    
    return pkg
end;

In [9]:
function package_shortterm()
    date_range = sort(unique(shortterm.date))
    
    d_dict = Dict((row.hospital, row.date) => row for row in eachrow(shortterm))
    
    active_total   = [haskey(d_dict, (h,t)) ? coalesce(d_dict[(h,t)].occupancy_total, NaN)  : NaN for h in hospitals, t in date_range]
    active_icu     = [haskey(d_dict, (h,t)) ? coalesce(d_dict[(h,t)].occupancy_icu, NaN)    : NaN for h in hospitals, t in date_range]
    active_acute   = [haskey(d_dict, (h,t)) ? coalesce(d_dict[(h,t)].occupancy_acute, NaN)  : NaN for h in hospitals, t in date_range]
    admitted_total = [haskey(d_dict, (h,t)) ? coalesce(d_dict[(h,t)].admissions_total, NaN) : NaN for h in hospitals, t in date_range]
    
    pkg = Dict()
    pkg[:total] = (active=active_total, admitted=admitted_total)
    pkg[:icu] = (active=active_icu,)
    pkg[:acute] = (active=active_acute,)
    pkg[:meta] = (;hospitals, date_range)
    
    return pkg
end;

In [10]:
function package_longterm()
    patienttypes = unique(longterm.patienttype)
    scenarios = unique(longterm.scenario)
    
    date_range = sort(unique(longterm.date))
    
    pkg = Dict()
    for pt in patienttypes, s in scenarios
        d = filter(r -> r.patienttype == pt && r.scenario == s, longterm)
        d_dict = Dict((row.hospital, row.date) => row for row in eachrow(d))
        active       = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].active       : NaN for h in hospitals, t in date_range]
        active_std   = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].active_std   : NaN for h in hospitals, t in date_range]
        admitted     = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].admitted     : NaN for h in hospitals, t in date_range]
        admitted_std = [haskey(d_dict, (h,t)) ? d_dict[(h,t)].admitted_std : NaN for h in hospitals, t in date_range]
        pkg[(Symbol(pt),Symbol(s))] = (;active, active_std, admitted, admitted_std)
    end
    
    pkg[:meta] = (;patienttypes, scenarios, hospitals, date_range)
    
    return pkg
end;

In [11]:
function package_capacity(patienttype, levels)
    if patienttype == :icu
        hospitals = sort(unique(capacity.hospital))
        d_dict = Dict(row.hospital => row for row in eachrow(capacity))
        return [d_dict[h]["icu_covid_$l"] for h in hospitals, l in levels]
    elseif patienttype == :acute
        hospitals = sort(unique(capacity.hospital))
        d_dict = Dict(row.hospital => row for row in eachrow(capacity))
        return [d_dict[h]["ward_covid_$l"] for h in hospitals, l in levels]
    elseif patienttype == :total
        c1 = package_capacity(:icu, levels)
        c2 = package_capacity(:acute, levels)
        return c1 + c2
    else
        error("Invalid patienttype: $(patienttype)")
    end
end;

function package_capacity()
    levels = unique([split(l, "_")[end] for l in setdiff(names(capacity), ["hospital"])])
    hospitals = sort(unique(capacity.hospital))
    
    pkg = Dict()
    for pt in [:icu, :acute, :total]
        pkg[pt] = package_capacity(pt, levels)
    end
    pkg[:meta] = (;hospitals, capacity_names=["Baseline", "Ramp-Up", "Surge", "Surge+", "Max", "Crisis"])
    
    return pkg
end;

In [12]:
outdata = (
    realdata  = package_realdata(),
    shortterm = package_shortterm(),
    longterm  = package_longterm(),
    capacity  = package_capacity(),
);

In [13]:
serialize("../data/alldata.jlser", outdata);